# LAB 6: XÂY DỰNG MẠNG NƠ-RON SÂU (DEEP NEURAL NETWORK)
**Bài toán:** Nhận dạng quần áo, giày dép thời trang với bộ dữ liệu **Fashion-MNIST**

**Thư viện:** PyTorch

## Bước 0 – Kiểm tra GPU & Import thư viện

In [ ]:
# Kiểm tra GPU (vào Runtime → Change runtime type → T4 GPU)
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Thiết bị đang dùng: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ Không có GPU – vào Runtime → Change runtime type → T4 GPU')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import datasets

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Seed để tái lập kết quả
torch.manual_seed(42)
np.random.seed(42)

print('✅ Import thư viện thành công!')
print(f'PyTorch version: {torch.__version__}')

---
## Bước 1 – Tải và khám phá dữ liệu Fashion-MNIST

In [ ]:
data_folder = '~/data/FMNIST'

# Tải dữ liệu train và validation
fmnist_train = datasets.FashionMNIST(data_folder, download=True, train=True)
fmnist_val   = datasets.FashionMNIST(data_folder, download=True, train=False)

tr_images  = fmnist_train.data
tr_targets = fmnist_train.targets
val_images  = fmnist_val.data
val_targets = fmnist_val.targets

print('=== THÔNG TIN DỮ LIỆU ===')
print(f'Train: {tr_images.shape}  |  Validation: {val_images.shape}')
print(f'Số lớp: {len(fmnist_train.classes)}')
print(f'Tên lớp: {fmnist_train.classes}')

In [ ]:
# Trực quan hóa mẫu dữ liệu: 10 lớp × 10 ảnh
class_names = fmnist_train.classes
R, C = 10, 10
fig, axes = plt.subplots(R, C, figsize=(12, 12))
fig.suptitle('Mẫu dữ liệu Fashion-MNIST (10 lớp × 10 ảnh)', fontsize=14, y=1.01)

for label_class, plot_row in enumerate(axes):
    label_x_rows = np.where(tr_targets == label_class)[0]
    for plot_cell in plot_row:
        plot_cell.grid(False)
        plot_cell.axis('off')
        ix = np.random.choice(label_x_rows)
        plot_cell.imshow(tr_images[ix], cmap='gray')
    plot_row[0].set_ylabel(class_names[label_class], fontsize=9, rotation=90)

plt.tight_layout()
plt.show()

---
## Bước 2 – Xây dựng Dataset & DataLoader

In [ ]:
class FMNISTDataset(Dataset):
    """
    Dataset tùy chỉnh cho Fashion-MNIST
    - Chuẩn hóa ảnh về [0, 1] bằng cách chia 255
    - Flatten ảnh 28x28 thành vector 784 chiều
    """
    def __init__(self, x, y):
        x = x.float() / 255.0          # Chuẩn hóa pixel về [0, 1]
        x = x.view(-1, 28 * 28)        # Flatten: (N, 28, 28) → (N, 784)
        self.x, self.y = x, y

    def __getitem__(self, ix):
        return self.x[ix].to(device), self.y[ix].to(device)

    def __len__(self):
        return len(self.x)


def get_data(batch_size=64):
    """Tạo DataLoader cho train và validation"""
    train_ds = FMNISTDataset(tr_images, tr_targets)
    val_ds   = FMNISTDataset(val_images, val_targets)

    trn_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds,   batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl


trn_dl, val_dl = get_data(batch_size=64)
print(f'Số batch train: {len(trn_dl)}')
print(f'Số batch val  : {len(val_dl)}')

# Kiểm tra 1 batch
sample_x, sample_y = next(iter(trn_dl))
print(f'Kích thước 1 batch – X: {sample_x.shape}, Y: {sample_y.shape}')

---
## Bước 3 – Xây dựng mạng nơ-ron sâu (Deep Neural Network)

```
Input (784)
  ↓
Linear(784 → 512) → BatchNorm → ReLU → Dropout(0.3)
  ↓
Linear(512 → 256) → BatchNorm → ReLU → Dropout(0.3)
  ↓
Linear(256 → 128) → BatchNorm → ReLU → Dropout(0.2)
  ↓
Linear(128 → 64)  → BatchNorm → ReLU → Dropout(0.2)
  ↓
Linear(64  → 10)  (Output – 10 lớp)
```

In [ ]:
class DeepNeuralNetwork(nn.Module):
    """
    Mạng nơ-ron sâu 4 lớp ẩn với:
    - Batch Normalization: ổn định quá trình huấn luyện
    - Dropout: chống overfitting
    - ReLU Activation: giải quyết vanishing gradient
    """
    def __init__(self, input_size=784, num_classes=10):
        super(DeepNeuralNetwork, self).__init__()

        self.network = nn.Sequential(
            # Lớp 1: 784 → 512
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(p=0.3),

            # Lớp 2: 512 → 256
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p=0.3),

            # Lớp 3: 256 → 128
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.2),

            # Lớp 4: 128 → 64
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.2),

            # Output: 64 → 10
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)


def get_model(lr=1e-3):
    """Khởi tạo model, loss function và optimizer"""
    model     = DeepNeuralNetwork().to(device)
    loss_fn   = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    return model, loss_fn, optimizer


# In kiến trúc mạng
model, loss_fn, optimizer = get_model()
print('=== KIẾN TRÚC MẠNG NƠ-RON SÂU ===')
print(model)

# Tổng số tham số
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTổng tham số    : {total_params:,}')
print(f'Tham số huấn luyện: {train_params:,}')

---
## Bước 4 – Định nghĩa hàm train, validate và accuracy

In [ ]:
def train_batch(x, y, model, optimizer, loss_fn):
    """Huấn luyện trên 1 batch"""
    model.train()                          # Bật chế độ train (Dropout & BN hoạt động)
    optimizer.zero_grad()                  # Xóa gradient cũ
    prediction = model(x)                  # Forward pass
    batch_loss = loss_fn(prediction, y)    # Tính loss
    batch_loss.backward()                  # Backward pass
    optimizer.step()                       # Cập nhật trọng số
    return batch_loss.item()


@torch.no_grad()
def accuracy(x, y, model):
    """Tính độ chính xác trên 1 batch"""
    model.eval()                           # Tắt Dropout & BN khi đánh giá
    prediction  = model(x)
    _, argmaxes = prediction.max(-1)
    is_correct  = (argmaxes == y)
    return is_correct.cpu().numpy().tolist()


@torch.no_grad()
def compute_val_loss(x, y, model, loss_fn):
    """Tính validation loss"""
    model.eval()
    prediction = model(x)
    return loss_fn(prediction, y).item()


print('✅ Định nghĩa hàm thành công!')

---
## Bước 5 – Huấn luyện mạng nơ-ron sâu với Early Stopping

In [ ]:
# ──────────────────────────────────────────────
# Cấu hình huấn luyện
# ──────────────────────────────────────────────
NUM_EPOCHS    = 20
PATIENCE      = 5        # Early stopping: dừng nếu val_loss không giảm sau 5 epoch
LEARNING_RATE = 1e-3

# Khởi tạo
trn_dl, val_dl = get_data(batch_size=64)
model, loss_fn, optimizer = get_model(lr=LEARNING_RATE)

# Learning rate scheduler: giảm LR khi val_loss không cải thiện
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                               patience=3, verbose=True)

# Lưu lịch sử huấn luyện
train_losses, train_accuracies = [], []
val_losses,   val_accuracies   = [], []

# Early stopping
best_val_loss  = float('inf')
patience_count = 0
best_weights   = None

print(f'Bắt đầu huấn luyện | Device: {device} | Epochs: {NUM_EPOCHS} | LR: {LEARNING_RATE}')
print('-' * 75)
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>10} | {"Val Loss":>9} | {"Val Acc":>8} | {"LR":>8}')
print('-' * 75)

for epoch in range(1, NUM_EPOCHS + 1):

    # ── TRAIN ──
    epoch_losses, epoch_accs = [], []
    for x, y in trn_dl:
        loss = train_batch(x, y, model, optimizer, loss_fn)
        epoch_losses.append(loss)

    for x, y in trn_dl:
        epoch_accs.extend(accuracy(x, y, model))

    train_loss = np.mean(epoch_losses)
    train_acc  = np.mean(epoch_accs)

    # ── VALIDATE ──
    val_accs_list = []
    for x, y in val_dl:
        val_accs_list.extend(accuracy(x, y, model))
        v_loss = compute_val_loss(x, y, model, loss_fn)

    val_acc  = np.mean(val_accs_list)
    val_loss = v_loss

    # Lưu lịch sử
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    # Learning rate scheduler
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f'{epoch:>6} | {train_loss:>10.4f} | {train_acc:>9.2%} | {val_loss:>9.4f} | {val_acc:>7.2%} | {current_lr:>8.6f}')

    # ── EARLY STOPPING ──
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_count = 0
        # Lưu trọng số tốt nhất
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'\n⏹️  Early stopping tại epoch {epoch} | Best val_loss: {best_val_loss:.4f}')
            break

# Nạp lại trọng số tốt nhất
model.load_state_dict(best_weights)
print(f'\n✅ Hoàn thành! Best val_loss: {best_val_loss:.4f}')

---
## Bước 6 – Trực quan hóa kết quả huấn luyện

In [ ]:
epochs_range = np.arange(1, len(train_losses) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Kết quả huấn luyện Deep Neural Network – Fashion-MNIST', fontsize=13)

# Loss
axes[0].plot(epochs_range, train_losses, 'bo-', label='Train Loss',      markersize=5)
axes[0].plot(epochs_range, val_losses,   'r-',  label='Validation Loss', linewidth=2)
axes[0].set_title('Loss qua các epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(2))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, train_accuracies, 'bo-', label='Train Accuracy',      markersize=5)
axes[1].plot(epochs_range, val_accuracies,   'r-',  label='Validation Accuracy', linewidth=2)
axes[1].set_title('Accuracy qua các epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].xaxis.set_major_locator(mticker.MultipleLocator(2))
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n📊 Kết quả cuối cùng:')
print(f'   Train Accuracy    : {train_accuracies[-1]:.2%}')
print(f'   Validation Accuracy: {val_accuracies[-1]:.2%}')
print(f'   Best Val Loss     : {best_val_loss:.4f}')

---
## Bước 7 – Đánh giá chi tiết: Confusion Matrix & Classification Report

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Thu thập nhãn dự đoán và nhãn thật trên tập validation
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for x, y in val_dl:
        outputs = model(x)
        _, preds = outputs.max(-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix – Validation Set', fontsize=13)
plt.ylabel('Nhãn thật (True Label)')
plt.xlabel('Nhãn dự đoán (Predicted Label)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
print('=== CLASSIFICATION REPORT ===')
print(classification_report(all_labels, all_preds, target_names=class_names))

---
## Bước 8 – Trực quan hóa kết quả dự đoán

In [ ]:
# Hiển thị 25 ảnh ngẫu nhiên cùng nhãn dự đoán vs nhãn thật
val_images_flat = val_images.float() / 255.0

n_samples = 25
indices   = np.random.choice(len(val_images), n_samples, replace=False)

fig, axes = plt.subplots(5, 5, figsize=(13, 13))
fig.suptitle('Kết quả dự đoán trên tập Validation\n(Xanh = Đúng | Đỏ = Sai)', fontsize=13)

model.eval()
with torch.no_grad():
    for ax, idx in zip(axes.flat, indices):
        img   = val_images_flat[idx].view(1, -1).to(device)
        label = val_targets[idx].item()

        output = model(img)
        proba  = F.softmax(output, dim=-1)
        pred   = output.argmax(-1).item()
        conf   = proba[0, pred].item()

        ax.imshow(val_images[idx].numpy(), cmap='gray')
        ax.axis('off')

        color = '#2ecc71' if pred == label else '#e74c3c'
        ax.set_title(
            f'Thật: {class_names[label]}\nDự đoán: {class_names[pred]} ({conf:.0%})',
            fontsize=7, color=color, fontweight='bold'
        )

plt.tight_layout()
plt.show()

---
## Bước 9 – So sánh Lab 5 (Shallow NN) vs Lab 6 (Deep NN)

In [ ]:
# ──────────────────────────────────────────────────────
# Xây dựng lại mô hình Lab 5 để so sánh công bằng
# ──────────────────────────────────────────────────────
print('Đang huấn luyện mô hình Lab 5 (Shallow NN) để so sánh...')

shallow_model = nn.Sequential(
    nn.Linear(28 * 28, 1000),
    nn.ReLU(),
    nn.Linear(1000, 10)
).to(device)

shallow_loss_fn   = nn.CrossEntropyLoss()
shallow_optimizer = Adam(shallow_model.parameters(), lr=1e-3)

shallow_train_acc_hist, shallow_val_acc_hist = [], []

for epoch in range(1, len(train_losses) + 1):
    # Train
    for x, y in trn_dl:
        shallow_model.train()
        shallow_optimizer.zero_grad()
        pred = shallow_model(x)
        loss = shallow_loss_fn(pred, y)
        loss.backward()
        shallow_optimizer.step()

    # Train accuracy
    s_train_accs = []
    shallow_model.eval()
    with torch.no_grad():
        for x, y in trn_dl:
            _, p = shallow_model(x).max(-1)
            s_train_accs.extend((p == y).cpu().numpy().tolist())

    # Val accuracy
    s_val_accs = []
    with torch.no_grad():
        for x, y in val_dl:
            _, p = shallow_model(x).max(-1)
            s_val_accs.extend((p == y).cpu().numpy().tolist())

    shallow_train_acc_hist.append(np.mean(s_train_accs))
    shallow_val_acc_hist.append(np.mean(s_val_accs))

print('✅ Xong!')

In [ ]:
# Biểu đồ so sánh accuracy
epochs_range = np.arange(1, len(train_losses) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('So sánh Lab 5 (Shallow NN) vs Lab 6 (Deep NN)', fontsize=13)

# Train accuracy
axes[0].plot(epochs_range, shallow_train_acc_hist, 'b--o', label='Lab5 – Shallow NN', markersize=4, alpha=0.7)
axes[0].plot(epochs_range, train_accuracies,        'r-o',  label='Lab6 – Deep NN',   markersize=4)
axes[0].set_title('Train Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation accuracy
axes[1].plot(epochs_range, shallow_val_acc_hist, 'b--o', label='Lab5 – Shallow NN', markersize=4, alpha=0.7)
axes[1].plot(epochs_range, val_accuracies,        'r-o',  label='Lab6 – Deep NN',   markersize=4)
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Bảng so sánh tóm tắt
print('\n=== BẢNG SO SÁNH KẾT QUẢ CUỐI CÙNG ===')
print(f'{"Mô hình":<25} {"Train Acc":>12} {"Val Acc":>12}')
print('-' * 50)
print(f'{"Lab5 – Shallow NN":<25} {shallow_train_acc_hist[-1]:>11.2%} {shallow_val_acc_hist[-1]:>11.2%}')
print(f'{"Lab6 – Deep NN":<25} {train_accuracies[-1]:>11.2%} {val_accuracies[-1]:>11.2%}')
print(f'\n📈 Cải thiện val accuracy: +{(val_accuracies[-1] - shallow_val_acc_hist[-1]):.2%}')

---
## Bước 10 – Lưu model

In [ ]:
# Lưu trọng số model tốt nhất
torch.save(model.state_dict(), 'lab6_deep_nn_fmnist.pth')
print('✅ Đã lưu model: lab6_deep_nn_fmnist.pth')

# Tải lại model (ví dụ dùng cho inference)
loaded_model = DeepNeuralNetwork().to(device)
loaded_model.load_state_dict(torch.load('lab6_deep_nn_fmnist.pth'))
loaded_model.eval()
print('✅ Tải lại model thành công!')

# Download file về máy
from google.colab import files
files.download('lab6_deep_nn_fmnist.pth')